<a href="https://colab.research.google.com/github/anshupandey/EY_GenAI_Architect/blob/main/Lab1_3_LLM_output_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1.3 — Output Control & Reliability

**Module 1 | LLM Foundations + Model Selection + Scalable GenAI**

---

## What you will learn

By the end of this notebook you will be able to:

1. Understand the five-layer reliability stack for LLM output control
2. Harden system prompts to establish an explicit output contract (Layer 1)
3. Use few-shot examples as a structural anchor for output shape (Layer 2)
4. Enforce valid JSON using `json_object` mode (Layer 3)
5. Enforce exact field names, types and enums using `json_schema` mode (Layer 4a)
6. Use `client.responses.parse()` with Pydantic for typed Python objects (Layer 4b)
7. Add field validators and cross-field business rules to Pydantic models
8. Embed chain-of-thought reasoning inside a structured output schema
9. Build retry and fallback patterns for production resilience (Layer 5)

---

## The reliability stack

Output control is a **layered stack**. Each layer adds reliability at a cost.

```
┌─────────────────────────────────────────────────────────────────┐
│  Layer 5 │ Retry + fallback              (application code)     │
│  Layer 4 │ Pydantic / JSON Schema        (API-level, strict)    │
│  Layer 3 │ json_object mode              (valid JSON, no schema) │
│  Layer 2 │ Few-shot examples             (structural anchor)    │
│  Layer 1 │ System prompt hardening       (prompt engineering)   │
└─────────────────────────────────────────────────────────────────┘
            ↑ Apply progressively as reliability needs increase
```

- **Quick tasks / prototypes** → Layer 1 is enough
- **Production pipelines** → Layers 1 + 4 (Pydantic)
- **High-stakes automation** → All five layers

---

## Prerequisites

Complete Lab 1.1 and 1.2 first. This notebook uses the same client setup as Lab 1.1
and directly addresses the non-determinism failure mode from Lab 1.2.

## Setup

In [1]:
%pip install openai pydantic python-dotenv matplotlib --quiet

In [2]:
import os, json, time, re, random, textwrap
from enum import Enum
from typing import Optional, Literal
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIConnectionError, APIStatusError
from pydantic import BaseModel, Field, field_validator, model_validator

load_dotenv(override=True)

ENDPOINT   = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
API_KEY    = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")


BASE_URL   = f"{ENDPOINT}/openai/v1/"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# ── Helpers ───────────────────────────────────────────────────────────────────
def section(title):
    print(f"\n{'━'*65}\n  {title}\n{'━'*65}")

def try_parse_json(text: str):
    """Try to parse as JSON. Returns (success, parsed_or_None)."""
    clean = text.strip()
    for prefix in ["```json", "```"]:
        if clean.startswith(prefix):
            clean = clean[len(prefix):]
    if clean.endswith("```"):
        clean = clean[:-3]
    clean = clean.strip()
    try:
        return True, json.loads(clean)
    except json.JSONDecodeError:
        return False, None

print("✅ Setup complete")
print(f"   Deployment: {DEPLOYMENT}")
print(f"   Base URL  : {BASE_URL}")

✅ Setup complete
   Deployment: gpt-4.1-mini
   Base URL  : https://anshumf.openai.azure.com/openai/v1/


---

## Section 1 — The Baseline Problem

We use a consistent realistic task throughout this notebook: **extracting structured data
from an unstructured quarterly report**. This mirrors what analysts do every day.

In [3]:
# A realistic quarterly report — this is our source document for all exercises
SAMPLE_REPORT = """
Q3 2024 Performance Review — APAC Region

Total revenue for the quarter reached $47.3 million, representing a 12% increase
year-over-year against the $42.2 million recorded in Q3 2023. Operating margin
improved to 18.4% from 15.1% last year, driven primarily by efficiency gains in
the logistics division. Customer acquisition cost fell by 8% to $340 per customer.

Key risks include increasing competition in the Southeast Asian market, where three
new entrants launched in the period. Supply chain disruptions remain a medium-term
concern. The leadership team rates overall performance as Strong.

Recommended next steps: expand marketing investment in Vietnam and Thailand,
review supplier contracts by end of Q4 2024, and schedule a risk review for the
board meeting in November.
"""

EXTRACTION_PROMPT = f"Extract the key metrics and findings from this report:\n{SAMPLE_REPORT}"

# ─── Baseline: no format controls ───────────────────────────────────────────
section("BASELINE — Uncontrolled output (no format enforcement)")

parse_results = []
for i in range(3):
    resp = client.responses.create(
        model=DEPLOYMENT,
        input=EXTRACTION_PROMPT,
        temperature=0.7,
        max_output_tokens=300,
    )
    text = resp.output_text.strip()
    ok, _ = try_parse_json(text)
    parse_results.append(ok)
    print(f"\nRun {i+1} (first 200 chars):")
    print(f"  {text[:200]}")
    print(f"  → Parseable as JSON: {'✅' if ok else '❌'}")

print(f"\nJSON parse success rate: {sum(parse_results)}/3")
print("⚠️  Output format varies — impossible to build reliable downstream code on this.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BASELINE — Uncontrolled output (no format enforcement)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Run 1 (first 200 chars):
  Here are the key metrics and findings from the Q3 2024 Performance Review for the APAC Region:

**Key Metrics:**
- Total revenue: $47.3 million (12% increase year-over-year from $42.2 million in Q3 20
  → Parseable as JSON: ❌

Run 2 (first 200 chars):
  Here are the key metrics and findings from the Q3 2024 Performance Review for the APAC region:

**Key Metrics:**
- Total revenue: $47.3 million (12% increase year-over-year from $42.2 million in Q3 20
  → Parseable as JSON: ❌

Run 3 (first 200 chars):
  Here are the key metrics and findings from the Q3 2024 Performance Review for the APAC Region:

**Key Metrics:**
- Total revenue: $47.3 million (12% increase YoY from $42.2 million in Q3 2023)
- Opera
  → Parseable as JSON: ❌

JSON parse success rate: 0/3
⚠️  Output format 

---

## Section 2 — Layer 1: System Prompt Hardening

The cheapest reliability fix costs nothing — rewrite your `instructions`.

A well-hardened prompt gives the model an **explicit output contract**:
exact field names, exact format, what to do with missing values, and
negative constraints blocking the most common deviations.

This is **always the first layer to apply**, regardless of whether you add API-level
enforcement on top.

In [4]:
section("LAYER 1 — Vague vs hardened instructions")

VAGUE = "Extract the key metrics from the report."

HARDENED = """
You are a financial data extraction assistant.

TASK: Extract structured data from report text.

OUTPUT CONTRACT:
- Respond with valid JSON only. No prose, no markdown, no code fences.
- Use exactly these field names (no variations, no additions):
    quarter, region, revenue_usd_millions, revenue_growth_pct,
    operating_margin_pct, risks, performance_rating
- Numbers must be numeric types (float/int), NOT strings.
- Missing values → null.

NEGATIVE CONSTRAINTS:
- Do NOT add commentary, preamble, or explanation.
- Do NOT include fields not listed above.
- Do NOT wrap output in markdown code fences.
""".strip()

print("--- With VAGUE instructions (temperature=0) ---")
r1 = client.responses.create(
    model=DEPLOYMENT, instructions=VAGUE,
    input=EXTRACTION_PROMPT, temperature=0, max_output_tokens=300,
)
ok1, p1 = try_parse_json(r1.output_text)
print(r1.output_text[:300])
print(f"Parseable as JSON: {'✅' if ok1 else '❌'}")

print("\n--- With HARDENED instructions (temperature=0) ---")
r2 = client.responses.create(
    model=DEPLOYMENT, instructions=HARDENED,
    input=EXTRACTION_PROMPT, temperature=0, max_output_tokens=300,
)
ok2, p2 = try_parse_json(r2.output_text)
print(r2.output_text)
print(f"Parseable as JSON: {'✅' if ok2 else '❌'}")
if p2:
    print(f"Fields returned  : {list(p2.keys())}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 1 — Vague vs hardened instructions
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
--- With VAGUE instructions (temperature=0) ---
Key Metrics and Findings from Q3 2024 Performance Review — APAC Region:

- Total revenue: $47.3 million (12% increase year-over-year from $42.2 million in Q3 2023)
- Operating margin: 18.4% (up from 15.1% in Q3 2023)
- Customer acquisition cost: $340 per customer (8% decrease)
- Primary driver of ma
Parseable as JSON: ❌

--- With HARDENED instructions (temperature=0) ---
{
  "quarter": "Q3 2024",
  "region": "APAC",
  "revenue_usd_millions": 47.3,
  "revenue_growth_pct": 12.0,
  "operating_margin_pct": 18.4,
  "risks": "Increasing competition in the Southeast Asian market with three new entrants; supply chain disruptions remain a medium-term concern",
  "performance_rating": "Strong"
}
Parseable as JSON: ✅
Fields returned  : ['quarter', 'region', 'revenue_usd_millio

In [9]:
section("LAYER 1 — Hardened prompt anatomy")

print("""

SYSTEM PROMPT STRUCTURE FOR EXTRACTION TASKS
─────────────────────────────────────────────
  [ROLE]       → Sets domain context, improves extraction relevance
  [TASK]       → One clear sentence; prevents the model drifting to adjacent tasks
  [OUTPUT CONTRACT]
    - Format      → JSON only, no fences, no prose
    - Field names → Listed explicitly; model anchors to them
    - Types       → "Numbers must be float/int, not strings"
    - Nulls       → "If a field is missing from the text, use null"
  [NEGATIVE CONSTRAINTS]
    - "Do NOT add commentary"
    - "Do NOT include unlisted fields"
    - "Do NOT wrap in markdown code fences"
  [EDGE CASES]  (optional but valuable)
    - "If the text is not a report, return {"error": "not a report"}"

WHY NEGATIVES HELP
──────────────────
  The most common deviations are prose preamble, markdown code fences,
  and string-typed numbers. Naming them explicitly blocks them far more
  reliably than just saying "respond with JSON".
""")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 1 — Hardened prompt anatomy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


SYSTEM PROMPT STRUCTURE FOR EXTRACTION TASKS
─────────────────────────────────────────────
  [ROLE]       → Sets domain context, improves extraction relevance
  [TASK]       → One clear sentence; prevents the model drifting to adjacent tasks
  [OUTPUT CONTRACT]
    - Format      → JSON only, no fences, no prose
    - Field names → Listed explicitly; model anchors to them
    - Types       → "Numbers must be float/int, not strings"
    - Nulls       → "If a field is missing from the text, use null"
  [NEGATIVE CONSTRAINTS]
    - "Do NOT add commentary"
    - "Do NOT include unlisted fields"
    - "Do NOT wrap in markdown code fences"
  [EDGE CASES]  (optional but valuable)
    - "If the text is not a report, return {"error": "not a report"}"

WHY NEGATIVES HELP
──────────────────
  The most common deviations are prose 

---

## Section 3 — Layer 2: Few-Shot Examples

Few-shot examples show the model **exactly** what the output should look like.
This is the most underused reliability technique — and often more effective than
rewording instructions.

**Why it works:** The model's token prediction is anchored to the pattern it has
already "seen" multiple times. Deviating from an established pattern requires
sampling against the dominant probability mass.

In [10]:
section("LAYER 2 — Two-shot examples in the input")

# Build a multi-turn input with 2 example pairs before the real task
few_shot_input = [
    # Example 1: basic case
    {"role": "user",
     "content": "Extract from: 'Q1 2024 EMEA: Revenue $31.5M, up 9% YoY. Margin 14.2%. Rating: Satisfactory.'"},
    {"role": "assistant",
     "content": json.dumps({
         "quarter": "Q1 2024", "region": "EMEA",
         "revenue_usd_millions": 31.5, "revenue_growth_pct": 9.0,
         "operating_margin_pct": 14.2, "risks": [], "performance_rating": "Satisfactory"
     })},
    # Example 2: tests null handling (margin not available)
    {"role": "user",
     "content": "Extract from: 'Americas Q2 2024: $88.1M revenue, 21% growth. Supply constraints mentioned. No margin data.'"},
    {"role": "assistant",
     "content": json.dumps({
         "quarter": "Q2 2024", "region": "Americas",
         "revenue_usd_millions": 88.1, "revenue_growth_pct": 21.0,
         "operating_margin_pct": None, "risks": ["supply constraints"], "performance_rating": None
     })},
    # Real task
    {"role": "user", "content": EXTRACTION_PROMPT},
]

resp = client.responses.create(
    model=DEPLOYMENT,
    instructions="You extract financial metrics from report text. Respond only with valid JSON matching the schema in the examples.",
    input=few_shot_input,
    temperature=0, max_output_tokens=300,
)

ok, parsed = try_parse_json(resp.output_text)
print("Output (with 2 few-shot examples):")
print(resp.output_text)
print(f"\nParseable as JSON: {'✅' if ok else '❌'}")
if parsed:
    print(f"Fields present   : {list(parsed.keys())}")
    print(f"Null fields      : {[k for k,v in parsed.items() if v is None]}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 2 — Two-shot examples in the input
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Output (with 2 few-shot examples):
{
  "quarter": "Q3 2024",
  "region": "APAC",
  "revenue_usd_millions": 47.3,
  "revenue_growth_pct": 12.0,
  "operating_margin_pct": 18.4,
  "risks": [
    "increasing competition in the Southeast Asian market",
    "supply chain disruptions"
  ],
  "performance_rating": "Strong"
}

Parseable as JSON: ✅
Fields present   : ['quarter', 'region', 'revenue_usd_millions', 'revenue_growth_pct', 'operating_margin_pct', 'risks', 'performance_rating']
Null fields      : []


In [11]:
section("LAYER 2 — Anti-example pattern")

# Including a BAD example and correcting it teaches the model what NOT to do.
# Use this when you know the model's specific deviation pattern.

anti_ex_instructions = """
You extract financial metrics from report text.
Respond only with valid JSON.

BAD output (do NOT do this):
  Here are the extracted metrics: {"revenue": "$47.3M"}
  ↑ prose preamble is wrong; "$47.3M" is a string not a float

GOOD output (always do this):
  {"revenue_usd_millions": 47.3}
  ↑ pure JSON, numeric type, no preamble
""".strip()

resp = client.responses.create(
    model=DEPLOYMENT, instructions=anti_ex_instructions,
    input=EXTRACTION_PROMPT, temperature=0, max_output_tokens=300,
)
ok, _ = try_parse_json(resp.output_text)
print(resp.output_text[:300])
print(f"\nParseable as JSON: {'✅' if ok else '❌'}")
print("\n💡 Anti-examples are most effective when you already know the model's")
print("   specific failure pattern from production logs.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 2 — Anti-example pattern
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```json
{
  "quarter": "Q3 2024",
  "region": "APAC",
  "total_revenue_usd_millions": 47.3,
  "revenue_growth_percent_year_over_year": 12,
  "revenue_usd_millions_prior_year": 42.2,
  "operating_margin_percent": 18.4,
  "operating_margin_percent_prior_year": 15.1,
  "customer_acquisition_cost_usd": 

Parseable as JSON: ✅

💡 Anti-examples are most effective when you already know the model's
   specific failure pattern from production logs.


---

## Section 4 — Layer 3: JSON Object Mode

`json_object` mode is **API-level enforcement** that guarantees syntactically valid JSON.
It does not guarantee field names, types, or schema — just parseability.

```python
client.responses.create(
    ...
    text={"format": {"type": "json_object"}},   # ← enable JSON mode
)
```

> **Requirement:** Your `instructions` must mention "JSON" or the API will return an error.

In [13]:
section("LAYER 3 — json_object mode: guaranteed parseability")

parse_successes = 0
print("5 runs at temperature=0.9 WITH json_object mode:")
print("-" * 55)

for i in range(5):
    resp = client.responses.create(
        model=DEPLOYMENT,
        instructions="You are a data extraction assistant. Always respond with valid JSON.",
        input=EXTRACTION_PROMPT + "Provide the output in JSON format",
        temperature=0.9,          # high temperature — but JSON is enforced at API level
        max_output_tokens=300,
        text={"format": {"type": "json_object"}},
    )
    ok, parsed = try_parse_json(resp.output_text)
    parse_successes += int(ok)
    fields = list(parsed.keys()) if parsed else []
    print(f"  Run {i+1}: {'✅ Valid JSON' if ok else '❌ Invalid'} | fields: {fields}")

print(f"\nParse success: {parse_successes}/5")
print("✅ json_object mode guarantees parseability even at temperature=0.9.")
print("⚠️  Note: field names may still vary run-to-run — that is NOT enforced here.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 3 — json_object mode: guaranteed parseability
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5 runs at temperature=0.9 WITH json_object mode:
-------------------------------------------------------
  Run 1: ✅ Valid JSON | fields: ['report', 'metrics', 'key_risks', 'leadership_assessment', 'recommended_next_steps']
  Run 2: ✅ Valid JSON | fields: ['report_period', 'region', 'financial_metrics', 'key_risks', 'leadership_assessment', 'recommended_next_steps']
  Run 3: ✅ Valid JSON | fields: ['Q3_2024_Performance_Review_APAC']
  Run 4: ✅ Valid JSON | fields: ['report', 'metrics', 'key_findings', 'recommended_next_steps']
  Run 5: ✅ Valid JSON | fields: ['report_period', 'region', 'key_metrics', 'key_findings', 'recommended_next_steps']

Parse success: 5/5
✅ json_object mode guarantees parseability even at temperature=0.9.
⚠️  Note: field names may still vary run-to-run — that is NOT enforced here.

In [14]:
section("LAYER 3 — What json_object does and does NOT guarantee")

print("""
json_object GUARANTEES
──────────────────────
  ✅ Output is syntactically valid JSON (json.loads() will not raise)
  ✅ No prose preamble, markdown fences, or mixed content
  ✅ Works at any temperature

json_object does NOT guarantee
────────────────────────────────
  ❌ Specific field names  (may vary: "revenue" vs "total_revenue")
  ❌ Specific data types   (may return "47.3" string instead of 47.3 float)
  ❌ All required fields present
  ❌ No extra hallucinated fields

WHEN TO USE
───────────
  • Downstream code only needs json.loads() to succeed without crashing
  • Schema flexibility is acceptable
  • Quick reliability upgrade with minimal code changes
  • Prototypes and exploratory pipelines
""")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 3 — What json_object does and does NOT guarantee
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

json_object GUARANTEES
──────────────────────
  ✅ Output is syntactically valid JSON (json.loads() will not raise)
  ✅ No prose preamble, markdown fences, or mixed content
  ✅ Works at any temperature

json_object does NOT guarantee
────────────────────────────────
  ❌ Specific field names  (may vary: "revenue" vs "total_revenue")
  ❌ Specific data types   (may return "47.3" string instead of 47.3 float)
  ❌ All required fields present
  ❌ No extra hallucinated fields

WHEN TO USE
───────────
  • Downstream code only needs json.loads() to succeed without crashing
  • Schema flexibility is acceptable
  • Quick reliability upgrade with minimal code changes
  • Prototypes and exploratory pipelines



---

## Section 5 — Layer 4a: JSON Schema Mode

JSON Schema mode enforces a specific schema — field names, types, required fields,
and allowed enum values — at the API level using **constrained decoding**.
The model physically cannot produce tokens that violate the schema.

```python
client.responses.create(
    ...
    text={
        "format": {
            "type": "json_schema",
            "name": "MySchema",
            "schema": { ... },   # standard JSON Schema object
            "strict": True,      # no deviations allowed
        }
    },
)
```

In [15]:
section("LAYER 4a — Define and apply a JSON Schema")

# The schema specifies exactly what the output must look like
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "quarter":              {"type": ["string", "null"]},
        "region":               {"type": ["string", "null"]},
        "revenue_usd_millions": {"type": ["number", "null"]},
        "revenue_growth_pct":   {"type": ["number", "null"]},
        "operating_margin_pct": {"type": ["number", "null"]},
        "risks": {
            "type": "array",
            "items": {"type": "string"}
        },
        "performance_rating": {
            "type": ["string", "null"],
            "enum": ["Excellent", "Strong", "Satisfactory", "Below Target", "Critical", None]
        }
    },
    "required": [
        "quarter", "region", "revenue_usd_millions", "revenue_growth_pct",
        "operating_margin_pct", "risks", "performance_rating"
    ],
    "additionalProperties": False   # ← no extra fields allowed
}

resp = client.responses.create(
    model=DEPLOYMENT,
    instructions="Extract financial metrics from the report. Respond with JSON.",
    input=EXTRACTION_PROMPT,
    temperature=0,
    max_output_tokens=300,
    text={
        "format": {
            "type": "json_schema",
            "name": "ReportExtraction",
            "schema": REPORT_SCHEMA,
            "strict": True,
        }
    },
)

ok, parsed = try_parse_json(resp.output_text)
print("JSON Schema enforced output:")
print(json.dumps(parsed, indent=2))

required = set(REPORT_SCHEMA["required"])
rating_enum = REPORT_SCHEMA["properties"]["performance_rating"]["enum"]
print(f"\nAll required fields present : {'✅' if parsed and set(parsed.keys()) == required else '❌'}")
print(f"No extra fields             : {'✅' if parsed and set(parsed.keys()) == required else '❌'}")
print(f"Revenue is numeric          : {'✅' if parsed and isinstance(parsed.get('revenue_usd_millions'), (int,float)) else '❌'}")
print(f"Rating in enum              : {'✅' if parsed and parsed.get('performance_rating') in rating_enum else '❌'}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 4a — Define and apply a JSON Schema
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
JSON Schema enforced output:
{
  "quarter": "Q3 2024",
  "region": "APAC",
  "revenue_usd_millions": 47.3,
  "revenue_growth_pct": 12,
  "operating_margin_pct": 18.4,
  "risks": [
    "Increasing competition in the Southeast Asian market with three new entrants",
    "Supply chain disruptions as a medium-term concern"
  ],
  "performance_rating": "Strong"
}

All required fields present : ✅
No extra fields             : ✅
Revenue is numeric          : ✅
Rating in enum              : ✅


In [16]:
section("LAYER 4a — Schema compliance across 5 runs at temperature=1.0")

required = set(REPORT_SCHEMA["required"])
rating_enum = REPORT_SCHEMA["properties"]["performance_rating"]["enum"]

for i in range(5):
    resp = client.responses.create(
        model=DEPLOYMENT,
        instructions="Extract financial metrics from the report. Respond with JSON.",
        input=EXTRACTION_PROMPT,
        temperature=1.0,           # high temperature — schema still enforced
        max_output_tokens=300,
        text={"format": {"type": "json_schema", "name": "ReportExtraction",
                         "schema": REPORT_SCHEMA, "strict": True}},
    )
    ok, p = try_parse_json(resp.output_text)
    if p:
        fields_ok  = set(p.keys()) == required
        rating_ok  = p.get("performance_rating") in rating_enum
        numeric_ok = isinstance(p.get("revenue_usd_millions"), (int, float))
        print(f"Run {i+1}: JSON ✅ | Fields {'✅' if fields_ok else '❌'} | Rating enum {'✅' if rating_ok else '❌'} | Revenue numeric {'✅' if numeric_ok else '❌'}")
    else:
        print(f"Run {i+1}: ❌ Parse failed")

print("\n✅ json_schema strict mode maintains full compliance even at temperature=1.0.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 4a — Schema compliance across 5 runs at temperature=1.0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Run 1: JSON ✅ | Fields ✅ | Rating enum ✅ | Revenue numeric ✅
Run 2: JSON ✅ | Fields ✅ | Rating enum ✅ | Revenue numeric ✅
Run 3: JSON ✅ | Fields ✅ | Rating enum ✅ | Revenue numeric ✅
Run 4: JSON ✅ | Fields ✅ | Rating enum ✅ | Revenue numeric ✅
Run 5: JSON ✅ | Fields ✅ | Rating enum ✅ | Revenue numeric ✅

✅ json_schema strict mode maintains full compliance even at temperature=1.0.


---

## Section 6 — Layer 4b: Pydantic + `responses.parse()`

This is the **recommended default for production Python**. It combines API-level
schema enforcement with automatic deserialization into a typed Python object.

```python
# responses.create() — text output, you parse manually
resp = client.responses.create(...)
text = resp.output_text                    # str

# responses.parse() — typed Python object, no manual parsing
resp = client.responses.parse(..., text_format=MyModel)
obj  = resp.output_parsed                  # MyModel instance ← fully typed
```

Benefits over raw JSON Schema mode:
- **Typed Python object** — attribute access with IDE autocomplete
- **`Field(description=...)`** — field descriptions inform the model
- **`@field_validator`** — enforce business logic (ranges, format, conversions)
- **`@model_validator`** — cross-field rules
- **Enums** — native Python Enum support

In [17]:
section("LAYER 4b — Basic Pydantic model with responses.parse()")

class PerformanceRating(str, Enum):
    EXCELLENT    = "Excellent"
    STRONG       = "Strong"
    SATISFACTORY = "Satisfactory"
    BELOW_TARGET = "Below Target"
    CRITICAL     = "Critical"

class ReportExtraction(BaseModel):
    quarter:              Optional[str]              = Field(None, description="Fiscal quarter, e.g. Q3 2024")
    region:               Optional[str]              = Field(None, description="Geographic region")
    revenue_usd_millions: Optional[float]            = Field(None, description="Total revenue in USD millions")
    revenue_growth_pct:   Optional[float]            = Field(None, description="YoY revenue growth percentage")
    operating_margin_pct: Optional[float]            = Field(None, description="Operating margin as a percentage")
    risks:                list[str]                  = Field(default_factory=list, description="Identified risks")
    performance_rating:   Optional[PerformanceRating] = Field(None, description="Overall performance rating")

# Use responses.parse() — the Responses API native Pydantic path
resp = client.responses.parse(
    model=DEPLOYMENT,
    instructions="Extract financial metrics from the report.",
    input=[{"role": "user", "content": EXTRACTION_PROMPT}],
    text_format=ReportExtraction,           # ← pass the Pydantic class directly
    temperature=0,
    max_output_tokens=300,
)

report: ReportExtraction = resp.output_parsed   # fully typed object

print("Parsed Pydantic object:")
print(f"  quarter              : {report.quarter}")
print(f"  region               : {report.region}")
print(f"  revenue_usd_millions : {report.revenue_usd_millions}")
print(f"  revenue_growth_pct   : {report.revenue_growth_pct}")
print(f"  operating_margin_pct : {report.operating_margin_pct}")
print(f"  risks                : {report.risks}")
print(f"  performance_rating   : {report.performance_rating}")

print("\nType safety:")
print(f"  revenue is float   : {isinstance(report.revenue_usd_millions, float)}")
print(f"  rating is Enum     : {isinstance(report.performance_rating, PerformanceRating)}")
print(f"  risks is list      : {isinstance(report.risks, list)}")

if report.revenue_usd_millions:
    print(f"\nDirect computation (no parsing needed):")
    print(f"  Revenue in billions: ${report.revenue_usd_millions / 1000:.4f}B")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 4b — Basic Pydantic model with responses.parse()
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Parsed Pydantic object:
  quarter              : Q3 2024
  region               : APAC
  revenue_usd_millions : 47.3
  revenue_growth_pct   : 12.0
  operating_margin_pct : 18.4
  risks                : ['Increasing competition in the Southeast Asian market with three new entrants', 'Supply chain disruptions remain a medium-term concern']
  performance_rating   : PerformanceRating.STRONG

Type safety:
  revenue is float   : True
  rating is Enum     : True
  risks is list      : True

Direct computation (no parsing needed):
  Revenue in billions: $0.0473B


### 6.1 Nested Pydantic models

In [18]:
section("LAYER 4b — Nested models for structured sub-objects")

class RiskItem(BaseModel):
    description: str                          = Field(description="What the risk is")
    severity:    Literal["Low","Medium","High"] = Field(description="Risk severity level")

class NextStep(BaseModel):
    action:   str           = Field(description="Recommended action")
    deadline: Optional[str] = Field(None, description="Deadline if mentioned in the text")

class FullReportExtraction(BaseModel):
    quarter:              Optional[str]               = Field(None)
    region:               Optional[str]               = Field(None)
    revenue_usd_millions: Optional[float]             = Field(None)
    revenue_growth_pct:   Optional[float]             = Field(None)
    operating_margin_pct: Optional[float]             = Field(None)
    performance_rating:   Optional[PerformanceRating] = Field(None)
    risks:                list[RiskItem]              = Field(default_factory=list)
    next_steps:           list[NextStep]              = Field(default_factory=list)

resp = client.responses.parse(
    model=DEPLOYMENT,
    instructions="Extract all financial metrics, risks with severity, and next steps from the report.",
    input=[{"role": "user", "content": EXTRACTION_PROMPT}],
    text_format=FullReportExtraction,
    temperature=0,
    max_output_tokens=500,
)

full: FullReportExtraction = resp.output_parsed

print(f"Quarter : {full.quarter} | Region: {full.region}")
print(f"Revenue : ${full.revenue_usd_millions}M | Growth: {full.revenue_growth_pct}% | Margin: {full.operating_margin_pct}%")
print(f"Rating  : {full.performance_rating}")

print(f"\nRisks ({len(full.risks)}):")
for r in full.risks:
    print(f"  [{r.severity:<6}] {r.description}")

print(f"\nNext Steps ({len(full.next_steps)}):")
for ns in full.next_steps:
    dl = f" (by {ns.deadline})" if ns.deadline else ""
    print(f"  • {ns.action}{dl}")

# Type safety on nested objects
if full.risks:
    print(f"\nType check: full.risks[0] is RiskItem → {isinstance(full.risks[0], RiskItem)}")
    print(f"  .severity is Literal   → {full.risks[0].severity}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 4b — Nested models for structured sub-objects
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Quarter : Q3 2024 | Region: APAC
Revenue : $47.3M | Growth: 12.0% | Margin: 18.4%
Rating  : PerformanceRating.STRONG

Risks (2):
  [High  ] Increasing competition in the Southeast Asian market with three new entrants
  [Medium] Supply chain disruptions

Next Steps (3):
  • Expand marketing investment in Vietnam and Thailand
  • Review supplier contracts (by end of Q4 2024)
  • Schedule a risk review for the board meeting (by November)

Type check: full.risks[0] is RiskItem → True
  .severity is Literal   → High


### 6.2 Field validators — enforcing business rules

In [19]:
section("LAYER 4b — @field_validator and @model_validator")

class ValidatedReportExtraction(BaseModel):
    quarter:              Optional[str]              = Field(None)
    region:               Optional[str]              = Field(None)
    revenue_usd_millions: Optional[float]            = Field(None)
    revenue_growth_pct:   Optional[float]            = Field(None)
    operating_margin_pct: Optional[float]            = Field(None)
    performance_rating:   Optional[PerformanceRating] = Field(None)
    risks:                list[str]                  = Field(default_factory=list)

    # Revenue must be positive
    @field_validator("revenue_usd_millions")
    @classmethod
    def revenue_must_be_positive(cls, v):
        if v is not None and v <= 0:
            raise ValueError(f"revenue_usd_millions must be positive, got {v}")
        return v

    # Margin must be within a realistic range
    @field_validator("operating_margin_pct")
    @classmethod
    def margin_in_valid_range(cls, v):
        if v is not None and not (-100 <= v <= 100):
            raise ValueError(f"operating_margin_pct {v} is outside -100 to 100")
        return v

    # Normalise quarter format: "Q3-2024" → "Q3 2024"
    @field_validator("quarter")
    @classmethod
    def normalise_quarter_format(cls, v):
        if v is not None:
            v = re.sub(r"Q([1-4])[-/](\d{4})", r"Q\1 \2", v)
        return v

    # Cross-field: a rating only makes sense if we have revenue data
    @model_validator(mode="after")
    def rating_requires_revenue(self):
        if self.performance_rating and self.revenue_usd_millions is None:
            raise ValueError("Cannot have a performance_rating without revenue data")
        return self

# ── Test 1: valid data from actual report ────────────────────────────────────
print("Test 1: Valid extraction from report")
resp = client.responses.parse(
    model=DEPLOYMENT,
    instructions="Extract financial metrics from the report.",
    input=[{"role": "user", "content": EXTRACTION_PROMPT}],
    text_format=ValidatedReportExtraction,
    temperature=0, max_output_tokens=300,
)
r = resp.output_parsed
print(f"  ✅ Validation passed: quarter={r.quarter}, revenue=${r.revenue_usd_millions}M, margin={r.operating_margin_pct}%")

# ── Test 2: deliberately invalid — negative revenue ──────────────────────────
print("\nTest 2: Negative revenue (should be caught)")
try:
    ValidatedReportExtraction(quarter="Q3 2024", revenue_usd_millions=-5.0, risks=[])
except Exception as e:
    print(f"  ✅ Caught: {e}")

# ── Test 3: margin out of range ──────────────────────────────────────────────
print("\nTest 3: Margin out of range (should be caught)")
try:
    ValidatedReportExtraction(quarter="Q3 2024", operating_margin_pct=250.0, risks=[])
except Exception as e:
    print(f"  ✅ Caught: {e}")

# ── Test 4: quarter normalisation ────────────────────────────────────────────
print("\nTest 4: Quarter format normalisation")
r2 = ValidatedReportExtraction(quarter="Q3-2024", risks=[])
print(f"  Input 'Q3-2024' → normalised to '{r2.quarter}'")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 4b — @field_validator and @model_validator
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Test 1: Valid extraction from report
  ✅ Validation passed: quarter=Q3 2024, revenue=$47.3M, margin=18.4%

Test 2: Negative revenue (should be caught)
  ✅ Caught: 1 validation error for ValidatedReportExtraction
revenue_usd_millions
  Value error, revenue_usd_millions must be positive, got -5.0 [type=value_error, input_value=-5.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Test 3: Margin out of range (should be caught)
  ✅ Caught: 1 validation error for ValidatedReportExtraction
operating_margin_pct
  Value error, operating_margin_pct 250.0 is outside -100 to 100 [type=value_error, input_value=250.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Test 4: Quarter format normalisation
  Input 'Q3-20

### 6.3 Chain-of-thought inside a structured schema

In [20]:
section("LAYER 4b — Chain-of-thought reasoning field")

# Give the model a dedicated reasoning field BEFORE the answer fields.
# This improves accuracy on complex tasks and makes the output auditable.

class ReportSentiment(BaseModel):
    reasoning:      str                                  = Field(
        description="Step-by-step reasoning: analyse metrics, weigh risks against performance, then conclude."
    )
    sentiment:      Literal["Positive", "Neutral", "Negative"]
    confidence_pct: int                                  = Field(description="Confidence 0-100")
    key_evidence:   list[str]                            = Field(description="Top 3 data points supporting the verdict")

resp = client.responses.parse(
    model=DEPLOYMENT,
    instructions="Analyse the business performance in the report and determine overall sentiment.",
    input=[{"role": "user", "content": f"Report:\n{SAMPLE_REPORT}"}],
    text_format=ReportSentiment,
    temperature=0, max_output_tokens=400,
)

s: ReportSentiment = resp.output_parsed

print("Reasoning (the model's work — auditable):")
print("-" * 55)
for line in textwrap.wrap(s.reasoning, 65):
    print(f"  {line}")
print(f"\nSentiment    : {s.sentiment}")
print(f"Confidence   : {s.confidence_pct}%")
print(f"Key evidence :")
for e in s.key_evidence:
    print(f"  • {e}")
print("\n💡 The reasoning field shows exactly why the model reached its verdict.")
print("   This is far more auditable than a bare sentiment label.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 4b — Chain-of-thought reasoning field
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Reasoning (the model's work — auditable):
-------------------------------------------------------
  The report shows a 12% year-over-year revenue increase to $47.3
  million, indicating strong top-line growth. Operating margin
  improved significantly from 15.1% to 18.4%, reflecting better
  cost management and efficiency gains, especially in logistics.
  Customer acquisition cost decreased by 8%, suggesting improved
  marketing efficiency. However, there are noted risks such as
  increased competition in Southeast Asia and ongoing supply chain
  disruptions, which could impact future performance. The
  leadership team rates the performance as Strong, and proactive
  steps are planned to mitigate risks and capitalize on growth
  opportunities. Overall, the positive financial metrics and
  strategic actions out

---

## Section 7 — Comparison: All Three JSON Enforcement Modes

In [21]:
section("COMPARISON — json_object vs json_schema vs responses.parse() + Pydantic")

print("""
┌──────────────────────────┬────────────────┬────────────────┬────────────────────┐
│ Feature                  │  json_object   │  json_schema   │ responses.parse()  │
│                          │  (Layer 3)     │  (Layer 4a)    │ + Pydantic (4b)    │
├──────────────────────────┼────────────────┼────────────────┼────────────────────┤
│ Valid JSON guaranteed     │ ✅             │ ✅             │ ✅                 │
│ Required fields enforced  │ ❌             │ ✅             │ ✅                 │
│ Field names locked        │ ❌             │ ✅             │ ✅                 │
│ Data types enforced       │ ❌             │ ✅             │ ✅                 │
│ Enum constraints          │ ❌             │ ✅             │ ✅                 │
│ No extra fields           │ ❌             │ ✅ (strict)    │ ✅                 │
│ Typed Python object       │ ❌ dict only   │ ❌ dict only   │ ✅ full object     │
│ Field descriptions        │ ❌             │ ❌             │ ✅ Field()         │
│ Business-rule validators  │ ❌             │ ❌             │ ✅ @field_validator │
│ IDE autocomplete          │ ❌             │ ❌             │ ✅                 │
│ Schema lives in           │ Instructions   │ API parameter  │ Python class       │
└──────────────────────────┴────────────────┴────────────────┴────────────────────┘

DECISION GUIDE
──────────────
  json_object   → Downstream code just needs json.loads() to not crash;
                  schema flexibility is acceptable.

  json_schema   → Strict field compliance needed; schema is shared outside Python
                  or defined in config (not code).

  responses.parse() + Pydantic
                → Production Python pipelines. RECOMMENDED DEFAULT.
                  Gives typed objects, IDE support, and business validators.
""")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  COMPARISON — json_object vs json_schema vs responses.parse() + Pydantic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

┌──────────────────────────┬────────────────┬────────────────┬────────────────────┐
│ Feature                  │  json_object   │  json_schema   │ responses.parse()  │
│                          │  (Layer 3)     │  (Layer 4a)    │ + Pydantic (4b)    │
├──────────────────────────┼────────────────┼────────────────┼────────────────────┤
│ Valid JSON guaranteed     │ ✅             │ ✅             │ ✅                 │
│ Required fields enforced  │ ❌             │ ✅             │ ✅                 │
│ Field names locked        │ ❌             │ ✅             │ ✅                 │
│ Data types enforced       │ ❌             │ ✅             │ ✅                 │
│ Enum constraints          │ ❌             │ ✅             │ ✅                 │
│ No extra fields           │ ❌             │ 

---

## Section 8 — Layer 5: Retry & Fallback Patterns

Even with all layers applied, production systems fail: rate limits, timeouts,
content filter triggers, and edge-case inputs all cause errors. A robust pipeline
needs **retry** (for transient failures) and **fallback** (for structural failures).

| Failure type | Retryable? | Strategy |
|---|---|---|
| `RateLimitError` | ✅ Yes | Exponential backoff |
| `APIConnectionError` | ✅ Yes | Retry up to N times |
| `ValidationError` | ⚠️ Sometimes | Retry with error feedback appended |
| `status == "incomplete"` | ⚠️ Yes | Increase `max_output_tokens` |
| HTTP 400 / content filter | ❌ No | Return structured error, escalate |

In [22]:
section("LAYER 5 — Retry with exponential backoff")

from pydantic import ValidationError

def call_with_retry(
    input_text: str,
    pydantic_model,
    instructions: str,
    max_retries: int = 3,
    base_delay: float = 1.0,
    temperature: float = 0,
    max_tokens: int = 400,
):
    """
    Calls client.responses.parse() with exponential backoff.

    Retry strategy:
      RateLimitError / APIConnectionError → wait and retry with backoff
      ValidationError                     → retry once with error context appended
      HTTP 400/403 (content filter etc.)  → raise immediately (not retryable)
    """
    last_error = None
    validation_feedback = None

    for attempt in range(1, max_retries + 1):
        try:
            effective_input = input_text
            if validation_feedback:
                effective_input = (
                    f"{input_text}\n\n"
                    f"[Previous attempt failed validation: {validation_feedback}. "
                    f"Please correct the output to match the required schema.]"
                )

            resp = client.responses.parse(
                model=DEPLOYMENT,
                instructions=instructions,
                input=[{"role": "user", "content": effective_input}],
                text_format=pydantic_model,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )

            if resp.status == "incomplete":
                raise ValueError(f"Response truncated by max_output_tokens: {resp.incomplete_details}")

            if resp.output_parsed is None:
                raise ValueError("Model returned None — possible refusal.")

            print(f"  ✅ Success on attempt {attempt}")
            return resp.output_parsed

        except (RateLimitError, APIConnectionError) as e:
            last_error = e
            delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            print(f"  ⚠️  Attempt {attempt} — {type(e).__name__}. Retrying in {delay:.1f}s...")
            time.sleep(delay)

        except ValidationError as e:
            last_error = e
            validation_feedback = str(e)[:200]
            print(f"  ⚠️  Attempt {attempt} — ValidationError. Retrying with correction hint...")
            time.sleep(0.5)

        except APIStatusError as e:
            if e.status_code in (400, 403):
                print(f"  ❌ Non-retryable error (HTTP {e.status_code})")
                raise
            last_error = e
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  ⚠️  Attempt {attempt} — HTTP {e.status_code}. Retrying in {delay:.1f}s...")
            time.sleep(delay)

    print(f"  ❌ All {max_retries} attempts exhausted.")
    raise last_error


print("Calling with retry wrapper (normal successful case):")
result = call_with_retry(
    input_text=EXTRACTION_PROMPT,
    pydantic_model=ReportExtraction,
    instructions="Extract financial metrics from the report.",
)
print(f"  quarter={result.quarter}, revenue=${result.revenue_usd_millions}M")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 5 — Retry with exponential backoff
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Calling with retry wrapper (normal successful case):
  ✅ Success on attempt 1
  quarter=Q3 2024, revenue=$47.3M


In [23]:
section("LAYER 5 — Fallback: strict schema → relaxed schema")

# When the strict schema fails (e.g. ambiguous input), fall back to a simpler one.

class SimpleExtraction(BaseModel):
    """Fallback schema — fewer required fields, more flexible."""
    summary:          str            = Field(description="One sentence summary of key findings")
    revenue_found:    bool           = Field(description="Whether any revenue data was found")
    raw_metrics:      dict[str, str] = Field(default_factory=dict,
                                             description="Any numeric values found, as string pairs")

def extract_with_fallback(text: str) -> tuple[str, BaseModel]:
    """Try full schema first; fall back to simple schema on failure."""
    # Attempt 1: full structured extraction
    try:
        resp = client.responses.parse(
            model=DEPLOYMENT,
            instructions="Extract financial metrics from the report.",
            input=[{"role": "user", "content": f"Extract from:\n{text}"}],
            text_format=ReportExtraction,
            temperature=0, max_output_tokens=300,
        )
        if resp.output_parsed:
            return "FullSchema", resp.output_parsed
    except Exception as e:
        print(f"  Full schema failed: {type(e).__name__} — trying fallback...")

    # Attempt 2: simple fallback schema
    resp = client.responses.parse(
        model=DEPLOYMENT,
        instructions="Extract whatever financial information you can from this text.",
        input=[{"role": "user", "content": f"Extract from:\n{text}"}],
        text_format=SimpleExtraction,
        temperature=0, max_output_tokens=200,
    )
    return "SimpleSchema (fallback)", resp.output_parsed


# Test 1: full financial report → expect FullSchema
print("Test 1: Financial report (should use FullSchema)")
schema_used, result = extract_with_fallback(SAMPLE_REPORT)
print(f"  Schema used: {schema_used} | Type: {type(result).__name__}")

# Test 2: non-financial text → FullSchema may fail gracefully or return nulls
print("\nTest 2: Non-financial email (may trigger fallback)")
email = "Hi team, just confirming the board meeting is at 3pm on Friday. See you there!"
schema_used, result = extract_with_fallback(email)
print(f"  Schema used: {schema_used}")
print(f"  Result     : {result}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 5 — Fallback: strict schema → relaxed schema
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Test 1: Financial report (should use FullSchema)
  Schema used: FullSchema | Type: ReportExtraction

Test 2: Non-financial email (may trigger fallback)
  Schema used: FullSchema
  Result     : quarter=None region=None revenue_usd_millions=None revenue_growth_pct=None operating_margin_pct=None risks=[] performance_rating=None


In [24]:
section("LAYER 5 — Production-ready pipeline (all layers combined)")

import time as _time

def extract_report_metrics(report_text: str, max_retries: int = 3) -> dict:
    """
    Production-grade extraction pipeline combining all reliability layers:
      Layer 1: Hardened system prompt
      Layer 2: One few-shot example
      Layer 4b: Pydantic via responses.parse()
      Layer 5: Retry with backoff + structured error return
    """
    INSTRUCTIONS = """
You are a financial data extraction assistant.
Extract structured financial metrics from report text.
Respond only with the requested structured data — no commentary.
If a field cannot be determined from the text, use null.
    """.strip()

    # One-shot example in the input
    full_input = [
        {"role": "user",
         "content": "Extract from: 'Q1 EMEA: $31.5M revenue, +9% YoY, 14.2% margin. Rating: Satisfactory.'"},
        {"role": "assistant",
         "content": json.dumps({
             "quarter": None, "region": "EMEA",
             "revenue_usd_millions": 31.5, "revenue_growth_pct": 9.0,
             "operating_margin_pct": 14.2, "risks": [], "performance_rating": "Satisfactory"
         })},
        {"role": "user", "content": f"Extract from:\n{report_text}"},
    ]

    t0 = _time.time()
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            resp = client.responses.parse(
                model=DEPLOYMENT,
                instructions=INSTRUCTIONS,
                input=full_input,
                text_format=ReportExtraction,
                temperature=0, max_output_tokens=400,
            )
            if resp.status == "incomplete":
                raise ValueError("Response truncated")
            return {
                "status": "success", "attempts": attempt,
                "latency_ms": int((_time.time() - t0) * 1000),
                "input_tokens": resp.usage.input_tokens,
                "output_tokens": resp.usage.output_tokens,
                "data": resp.output_parsed.model_dump(),
            }
        except (RateLimitError, APIConnectionError) as e:
            last_error = e
            delay = 1.0 * (2 ** (attempt - 1))
            print(f"  Attempt {attempt}: {type(e).__name__} — retry in {delay:.0f}s")
            time.sleep(delay)
        except Exception as e:
            last_error = e
            print(f"  Attempt {attempt}: {type(e).__name__}: {str(e)[:80]}")
            if attempt < max_retries:
                time.sleep(0.5)

    return {
        "status": "failed", "attempts": max_retries,
        "latency_ms": int((_time.time() - t0) * 1000),
        "error": str(last_error), "data": None,
    }


print("Running production pipeline...")
result = extract_report_metrics(SAMPLE_REPORT)
print(f"\nStatus       : {result['status']}")
print(f"Attempts     : {result['attempts']}")
print(f"Latency      : {result['latency_ms']}ms")
print(f"Tokens       : {result['input_tokens']} in / {result['output_tokens']} out")
print("\nExtracted data:")
for k, v in result["data"].items():
    print(f"  {k:<30}: {v}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LAYER 5 — Production-ready pipeline (all layers combined)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Running production pipeline...

Status       : success
Attempts     : 1
Latency      : 1405ms
Tokens       : 601 in / 67 out

Extracted data:
  quarter                       : Q3 2024
  region                        : APAC
  revenue_usd_millions          : 47.3
  revenue_growth_pct            : 12.0
  operating_margin_pct          : 18.4
  risks                         : ['increasing competition in the Southeast Asian market', 'Supply chain disruptions']
  performance_rating            : PerformanceRating.STRONG


---

## Section 9 — Deterministic Classification Patterns

Not all reliability problems need schemas. For **classification tasks** — where the
output is one of a small set of values — simpler techniques often suffice.

In [27]:
section("CLASSIFICATION — Four approaches compared")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

task = "Classify the sentiment: 'The product is okay but delivery was slow and support was unhelpful.'"

approaches = [
    {
        "name": "① Unconstrained (temp=0.9)",
        "instructions": "Classify the sentiment.",
        "kwargs": {}, "temp": 0.9,
    },
    {
        "name": "② temperature=0 only",
        "instructions": "Classify the sentiment.",
        "kwargs": {}, "temp": 0.0,
    },
    {
        "name": "③ Constrained prompt + temp=0",
        "instructions": "Classify sentiment. Reply with ONE word only: Positive, Negative, or Mixed.",
        "kwargs": {}, "temp": 0.0,
    },
    {
        "name": "④ Enum schema (temp=0.9)",
        "instructions": "Classify sentiment. Respond with JSON.",
        "kwargs": {"text": {"format": {"type": "json_schema", "name": "Sentiment",
            "schema": {"type": "object",
                       "properties": {"sentiment": {"type": "string", "enum": ["Positive","Negative","Mixed"]}},
                       "required": ["sentiment"], "additionalProperties": False},
            "strict": True}}},
        "temp": 0.9,
    },
]

N = 4
all_results = {}
for ap in approaches:
    outputs = []
    for _ in range(N):
        r = client.responses.create(
            model=DEPLOYMENT,
            instructions=ap["instructions"],
            input=task,
            temperature=ap["temp"],
            max_output_tokens=40,
            **ap["kwargs"]
        )
        outputs.append(r.output_text.strip())
    all_results[ap["name"]] = outputs

# Print results
for name, outputs in all_results.items():
    unique = set(outputs)
    consistent = len(unique) == 1
    label_short = name.replace("\n", " ")
    print(f"\n{label_short}")
    for i, o in enumerate(outputs):
        print(f"  Run {i+1}: {o[:60]}")
    print(f"  → {'✅ Fully consistent' if consistent else f'⚠️  {len(unique)} unique values'}")

print("\n💡 Approach ④ (enum schema) gives the strongest guarantee with high temperature.")
print("   Approach ③ (constrained prompt + temp=0) is the lightest-weight alternative.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CLASSIFICATION — Four approaches compared
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

① Unconstrained (temp=0.9)
  Run 1: The sentiment of the sentence "The product is okay but deliv
  Run 2: The sentiment of the sentence is negative.
  Run 3: The sentiment of the statement is negative.
  Run 4: The sentiment of the statement is **negative**.
  → ⚠️  4 unique values

② temperature=0 only
  Run 1: The sentiment of the statement is negative.
  Run 2: The sentiment of the statement is negative.
  Run 3: The sentiment of the statement is negative.
  Run 4: The sentiment of the statement is negative.
  → ✅ Fully consistent

③ Constrained prompt + temp=0
  Run 1: Negative
  Run 2: Negative
  Run 3: Negative
  Run 4: Negative
  → ✅ Fully consistent

④ Enum schema (temp=0.9)
  Run 1: {"sentiment":"Negative"}
  Run 2: {"sentiment":"Negative"}
  Run 3: {"sentiment":"Negative"}
  Run 4: {"sentiment":"Negat

---

## Summary & Key Takeaways

### The reliability stack — when to use each layer

```
Layer 1  System prompt hardening                         Cost: free
         → Always apply. Reduces load on every other layer.
         → Explicit output contract + negative constraints + edge case rules

Layer 2  Few-shot examples                               Cost: extra tokens
         → Apply when format compliance is critical or the model keeps deviating.
         → 2–3 input/output pairs; include anti-examples for known failure patterns

Layer 3  json_object mode                                Cost: negligible
         → Apply when downstream code calls json.loads() and crashes on prose.
         → Guarantees: valid JSON. Does NOT enforce fields.

Layer 4a json_schema mode                                Cost: first-request latency
         → Apply when exact field names, types, and enums must be enforced.
         → Returns a raw dict. Best when schema is defined outside Python code.

Layer 4b responses.parse() + Pydantic                   Cost: same as 4a
         → The recommended default for production Python.
         → Typed objects, IDE support, field validators, business logic.

Layer 5  Retry + fallback                                Cost: latency on failure
         → Apply in any production pipeline.
         → Exponential backoff for transient errors.
         → Correction-hint retry for ValidationError.
         → Simpler fallback schema for structurally incompatible inputs.
```

### Which layer solves which problem

| Problem | Layer(s) to apply |
|---|---|
| Output format varies across runs | 1 (hardened prompt) + 3 (json_object) |
| Field names vary | 4a or 4b |
| Revenue returned as string `"$47.3M"` | 4b — `@field_validator` to strip + convert |
| Enum field has unexpected values | 4a/4b — enum in schema |
| Rate limit errors in production | 5 — exponential backoff |
| Complex nested output structure | 4b — nested Pydantic models |
| Reasoning needs to be auditable | 4b — dedicated `reasoning` field |
| Input is unexpected type (not a report) | 5 — fallback to simpler schema |

---

## Module 1 Complete

The three notebooks together form the **Module 1 learning arc**:

```
Notebook 1: Understand the tool
  → LLM mechanics, tokenisation, Responses API, context windows, stale knowledge

Notebook 2: See what breaks
  → Five failure modes with live triggers, detection signals, and mitigations

Notebook 3: Learn to control it  ← you are here
  → Five-layer output reliability stack, Pydantic schemas, retry patterns
```

Future modules apply these foundations to: **prompt frameworks**, **work design
patterns for analysts**, **evaluation and verification**, and **responsible AI
governance**.